In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors
import torch
X_train = torch.Tensor(X_train)
X_test = torch.Tensor(X_test)
y_train = torch.Tensor(y_train)
y_test = torch.Tensor(y_test)




In [ ]:
# 2. Create TensorDataset objects
from torch.utils.data import DataLoader, TensorDataset
train_dataset = TensorDataset(X_train,y_train)
test_dataset = TensorDataset(X_test,y_test)



In [ ]:
# 3. Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)



In [ ]:
# 4. Print shape of one batch
X_batch, y_batch = next(iter(train_loader))
print(f"Training batch input shape: {X_batch.shape}")
print(f"Training batch labels shape: {y_batch.shape}")



In [ ]:
# 5. Display sample images
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 4))
X_batch, y_batch = next(iter(train_loader))

for i in range(6):
    plt.subplot(2, 3, i + 1)
    plt.imshow(X_batch[i].squeeze(), cmap='gray')
    plt.title(f"Label: {y_batch[i].item()}")
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import torch.nn as nn
class NN4Layer(nn.Module):

    def __init__(self,hidden_dim):

        super(NN4Layer, self).__init__()
        # input_dim = num of features, hidden_dim = num of neurons
        self.layer1 = nn.LazyLinear(hidden_dim)
        # hidden_dim = num of neurons, Output for binary classification is 1
        self.layer2 = nn.LazyLinear(hidden_dim)
        self.layer3 = nn.LazyLinear(hidden_dim)
        self.layer4 = nn.LazyLinear(1)
        # non-linearity activation function
        self.relu = nn.ReLU()
        # output activation function
        self.sigmoid = nn.Sigmoid()

    # forward pass
    def forward(self, x):
        a1 = self.relu(self.layer1(x))
        a2 = self.relu(self.layer2(a1))
        a3 = self.relu(self.layer3(a2))
        a4 = self.layer4(a3)
        return a4

In [ ]:
# Task 2: Write your training loop here:
def train_one_epoch(model, optimizer, criterion, train_loader, device):
  # Set the model to training mode
  model.train()

  running_loss = 0.0

  for X_batch, y_batch in train_loader:
    # Move batch to the selected device
    X_batch = X_batch.flatten(start_dim = 1).to(device)              # shape: (batch_size, num_features)
    y_batch = y_batch.view(-1, 1).to(device) # shape: (batch_size, 1)

    # Forward pass (continuous output)
    outputs = model(X_batch)                  # shape: (batch_size, 1)
    loss = criterion(outputs, y_batch)

    # Backward pass & optimization
    optimizer.zero_grad()   # Clear previous gradients
    loss.backward()         # Compute gradients
    optimizer.step()        # Update model parameters

    running_loss += loss.item()

  # Average loss over all batches
  avg_loss = running_loss / len(train_loader)

  return avg_loss

In [ ]:
# Task 3: Write your validation loop here:
def validate(model, criterion, test_loader, device):
  # Set the model to evaluation mode
  model.eval()

  running_loss = 0.0

  with torch.no_grad():
    for X_batch, y_batch in test_loader:
      # Move data to device
      X_batch = X_batch.flatten(start_dim=1).to(device)               # shape: (batch_size, num_features)
      y_batch = y_batch.view(-1, 1).to(device)  # shape: (batch_size, 1)

      # Forward pass (continuous output)
      outputs = model(X_batch)                   # shape: (batch_size, 1)
      loss = criterion(outputs, y_batch)

      running_loss += loss.item()

  # Average loss over all batches
  avg_loss = running_loss / len(test_loader)

  return avg_loss

In [ ]:
# Task 4: Define device, model, loss, optimizer:
from torch.optim import AdamW
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
hidden_dim = 128
model = NN4Layer(hidden_dim).to(device)
# Define criterion (loss function)
criterion = nn.MSELoss()
# Define optimizer
learning_rate = 0.001
optimizer = AdamW(model.parameters(), learning_rate)


In [ ]:
# Task 5: Start training for 20 epochs:
num_epochs = 20
# Run Training
train_losses = []
val_losses = []

print('Starting Training...')
for epoch in range(num_epochs):
  # Train one epoch
  train_loss = train_one_epoch(model, optimizer, criterion, train_loader, device)

  # Validate
  val_loss = validate(model, criterion, test_loader, device)

  train_losses.append(train_loss)
  val_losses.append(val_loss)

  i
  print(
    f'Epoch [{epoch+1}/{num_epochs}], '
    f'Train Loss: {train_loss:.4f}, '
    f'Val Loss: {val_loss:.4f}'
  )

print('Training Complete!')

In [ ]:
# Task 1: Write your code here:
# Plotting results
plt.figure(figsize=(7, 5))

plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Task 2 (Bonus): Write your code here: